In [ ]:
"""
Figure 1: Spatial SIC Trends and March Sea Ice Extent Timeseries
==================================================================

Four-panel figure:
- Panel (a): March 1986 ice edge with 1979-2014 spatial trends
- Panel (b): March 2023 ice edge with 2015-2025 spatial trends  
- Panel (c): Pan-Arctic March SIE timeseries with split linear trends
- Panel (d): Greenland Sea March SIE timeseries with split linear trends

Version: 1.0.0
Last Modified: 2026-02-27
"""

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as dates
from matplotlib.colors import LinearSegmentedColormap
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as mticker
import cmocean
import rockhound as rh
from scipy import stats
import os
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Setup logging
import sys
sys.path.append('..')
from utils.logger import (setup_logger, log_data_loading, log_processing_step, 
                          log_output_file, log_completion, log_error)

logger = setup_logger('figure_01', config_path='../config.yaml')
start_time = datetime.now()

try:
    # ========================================================================
    # CONFIGURATION
    # ========================================================================
    
    logger.info("Loading configuration...")
    
    # File paths
    ERA5_PATH = '../../era5/era5_*_Arctic.nc'
    OSISAF_PATH = '../../osi-sea_ice_index/'
    
    # Create output directories
    OUTPUT_DIR = Path('./outputs/figures')
    PROCESSED_DIR = Path('./outputs/processed_data/figure_01')
    METHODS_DIR = Path('./outputs/methods')
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    METHODS_DIR.mkdir(parents=True, exist_ok=True)
    
    # Maximum extent years
    MAX_YEAR_P1 = 1986
    MAX_YEAR_P2 = 2023
    
    # Analysis periods
    PERIOD_1 = (1979, 2016)
    PERIOD_2 = (2016, 2025)
    BREAKPOINT_YEAR = 2016
    
    # Spatial extents
    TREND_EXTENT = {'lon_min': -90, 'lon_max': 90, 'lat_min': 60, 'lat_max': 90}
    MAP_EXTENT = [-30, 20, 65, 85]
    
    # Region definitions
    GREENLAND_SEA_COORDS = [(-22, 71), (-8.5, 71), (12, 79), (-21, 79), (-28, 73), (-22, 71)]
    VERTICES = {'NW': (-21, 79), 'NE': (12, 79), 'SE': (-8.5, 71), 
                'SW': (-22, 71), 'W_Mid': (-28, 73)}
    MOORINGS = {'F17': -8.0, 'F14': -6.5, 'F13b': -5.5, 'F13': -5.0,
                'F12': -4.0, 'F11': -3.0, 'F10': -2.0}
    MOORING_LAT = 79.0
    
    # Plotting parameters
    SIC_THRESHOLD = 0.15
    RELIEF_ALPHA = 0.5
    TREND_ALPHA = 0.7
    DPI = 600
    
    # Colors
    COLOR_BOUNDARY = 'black'
    COLOR_ICE_EDGE = 'darkblue'
    COLOR_ARCTIC = '#7B1FA2'      # Classy purple for Pan-Arctic
    COLOR_GS = '#C62828'          # Classy red for Greenland Sea  
    COLOR_DATA = '#808080'        # Gray for data points
    COLOR_BREAKPOINT = '#808080'  # Gray for breakpoint line
    CI_ALPHA = 0.15
    
    # Colormap for trends
    colors_trend = ['#2166ac', '#4393c3', '#92c5de', '#d1e5f0', 'white',
                    '#fddbc7', '#f4a582', '#d6604d', '#b2182b']
    cmap_trend = LinearSegmentedColormap.from_list('red_white_blue', colors_trend, N=256)
    
    # Configuration for timeseries
    CONFIG = {
        'show_uncertainty_bands': True,
        'show_rate_annotations': False,
        'uncertainty_method': 'bootstrap',
        'confidence_level': 0.95,
        'n_bootstrap': 1000,
        'season': 'winter'
    }
    
    logger.info(f"Periods: {PERIOD_1} and {PERIOD_2}")
    logger.info(f"Breakpoint year: {BREAKPOINT_YEAR}")
    
    # ========================================================================
    # FUNCTIONS
    # ========================================================================
    
    def calculate_spatial_trends_vectorized(data_array, time_dim='time'):
        """Calculate linear trends for each grid cell."""
        logger.debug("Calculating spatial trends (vectorized)...")
        
        time_values = data_array[time_dim].values
        years = pd.to_datetime(time_values).year.values
        time_numeric = years - years[0]
        n_times = len(time_numeric)
        
        data = data_array.values
        n_lat, n_lon = data.shape[1], data.shape[2]
        data_reshaped = data.reshape(n_times, -1)
        
        slope = np.full(n_lat * n_lon, np.nan)
        pvalue = np.full(n_lat * n_lon, np.nan)
        rsquared = np.full(n_lat * n_lon, np.nan)
        
        n_valid = np.sum(~np.isnan(data_reshaped), axis=0)
        valid_cells = n_valid >= 5
        
        logger.debug(f"Valid cells: {valid_cells.sum()} / {len(valid_cells)}")
        
        if valid_cells.sum() > 0:
            for i in np.where(valid_cells)[0]:
                series = data_reshaped[:, i]
                valid_mask = ~np.isnan(series)
                
                if valid_mask.sum() >= 5:
                    result = stats.linregress(time_numeric[valid_mask], series[valid_mask])
                    slope[i] = result.slope
                    pvalue[i] = result.pvalue
                    rsquared[i] = result.rvalue ** 2
        
        slope = slope.reshape(n_lat, n_lon)
        pvalue = pvalue.reshape(n_lat, n_lon)
        rsquared = rsquared.reshape(n_lat, n_lon)
        
        spatial_dims = [d for d in data_array.dims if d != time_dim]
        spatial_coords = {dim: data_array[dim] for dim in spatial_dims}
        
        slope_da = xr.DataArray(slope, coords=spatial_coords, dims=spatial_dims)
        pvalue_da = xr.DataArray(pvalue, coords=spatial_coords, dims=spatial_dims)
        rsquared_da = xr.DataArray(rsquared, coords=spatial_coords, dims=spatial_dims)
        
        logger.debug(f"Slope range: {np.nanmin(slope):.6f} to {np.nanmax(slope):.6f} yr⁻¹")
        
        return slope_da, pvalue_da, rsquared_da
    
    
    def bootstrap_trend_ci(x, y, n_bootstrap=1000, confidence=0.95):
        """Bootstrap confidence intervals for linear trend."""
        x = np.asarray(x)
        y = np.asarray(y)
        
        n = len(y)
        trends = []
        predictions = np.zeros((n_bootstrap, n))
        
        for i in range(n_bootstrap):
            indices = np.random.choice(n, size=n, replace=True)
            x_boot = x[indices]
            y_boot = y[indices]
            
            z = np.polyfit(x_boot, y_boot, 1)
            p = np.poly1d(z)
            
            trend = (p(x[-1]) - p(x[0])) / len(x)
            trends.append(trend)
            predictions[i, :] = p(x)
        
        alpha = 1 - confidence
        lower_percentile = (alpha / 2) * 100
        upper_percentile = (1 - alpha / 2) * 100
        
        lower_bound = np.percentile(predictions, lower_percentile, axis=0)
        upper_bound = np.percentile(predictions, upper_percentile, axis=0)
        trend_ci = (np.percentile(trends, lower_percentile),
                    np.percentile(trends, upper_percentile))
        
        return {'lower_bound': lower_bound, 'upper_bound': upper_bound, 'trend_ci': trend_ci}
    
    
    def mann_kendall_test(x, y):
        """Mann-Kendall trend test for monotonic trends."""
        y = np.asarray(y)
        n = len(y)
        s = 0
        
        for i in range(n-1):
            for j in range(i+1, n):
                s += np.sign(y[j] - y[i])
        
        var_s = n * (n - 1) * (2 * n + 5) / 18
        
        if s > 0:
            z = (s - 1) / np.sqrt(var_s)
        elif s < 0:
            z = (s + 1) / np.sqrt(var_s)
        else:
            z = 0
        
        p_value = 2 * (1 - stats.norm.cdf(abs(z)))
        tau = s / (0.5 * n * (n - 1))
        
        if p_value < 0.05:
            trend = 'increasing' if tau > 0 else 'decreasing'
        else:
            trend = 'no trend'
        
        if p_value < 0.001:
            significance = '***'
        elif p_value < 0.01:
            significance = '**'
        elif p_value < 0.05:
            significance = '*'
        else:
            significance = 'n.s.'
        
        return {'tau': tau, 'p_value': p_value, 'trend': trend, 'significance': significance}
    
    
    def compare_trends(x1, y1, x2, y2):
        """Test if two trends are significantly different (Chow test)."""
        x1, y1 = np.asarray(x1), np.asarray(y1)
        x2, y2 = np.asarray(x2), np.asarray(y2)
        
        slope1, intercept1, _, _, _ = stats.linregress(x1, y1)
        slope2, intercept2, _, _, _ = stats.linregress(x2, y2)
        
        y1_fit = slope1 * x1 + intercept1
        y2_fit = slope2 * x2 + intercept2
        rss1 = np.sum((y1 - y1_fit)**2)
        rss2 = np.sum((y2 - y2_fit)**2)
        rss_separate = rss1 + rss2
        
        x_pooled = np.concatenate([x1, x2])
        y_pooled = np.concatenate([y1, y2])
        slope_pooled, intercept_pooled, _, _, _ = stats.linregress(x_pooled, y_pooled)
        y_pooled_fit = slope_pooled * x_pooled + intercept_pooled
        rss_pooled = np.sum((y_pooled - y_pooled_fit)**2)
        
        n1, n2 = len(y1), len(y2)
        k = 2
        f_statistic = ((rss_pooled - rss_separate) / k) / (rss_separate / (n1 + n2 - 2*k))
        p_value = 1 - stats.f.cdf(f_statistic, k, n1 + n2 - 2*k)
        
        return {'f_statistic': f_statistic, 'p_value': p_value,
                'significant': p_value < 0.05, 'slope_diff': slope2 - slope1}
    
    
    def get_trend(x, y):
        """Calculate linear trend with statistics."""
        x, y = np.asarray(x), np.asarray(y)
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        trend = (p(x[-1]) - p(x[0])) / len(x)
        result = stats.linregress(x, y)
        return p, trend, {'r_squared': result.rvalue**2, 'p_value': result.pvalue}
    
    
    def add_months(ds):
        """Add month coordinate to xarray dataset."""
        ds.coords['month'] = xr.DataArray(np.zeros(ds.time.shape), coords={'time': ds.time})
        for i, t in enumerate(ds.time):
            t = t.data
            month = t.astype('M8[M]').astype(int) % 12 + 1
            ds.coords['month'][i] = month
        return ds
    
    
    def fetch_sie(region, sie_list):
        """Fetch sea ice extent data for specified region."""
        for sie in sie_list:
            if sie.name == region:
                return sie
        raise ValueError(f"Region '{region}' not found in sie_list")
    
    logger.info("Functions defined")
    
    # ========================================================================
    # PART 1: CALCULATE SPATIAL TRENDS
    # ========================================================================
    
    log_processing_step(logger, "PART 1: Calculating Spatial Trends")
    
    # Load ERA5 data
    log_data_loading(logger, 'ERA5', ERA5_PATH)
    ds_era5 = xr.open_mfdataset(ERA5_PATH, combine='by_coords')
    if 'valid_time' in ds_era5:
        ds_era5 = ds_era5.rename({'valid_time': 'time'})
    
    logger.info(f"Time range: {ds_era5.time.min().dt.year.item()} to "
                f"{ds_era5.time.max().dt.year.item()}")
    logger.info(f"Grid: {len(ds_era5.latitude)} x {len(ds_era5.longitude)}")
    
    # Subset to trend calculation extent
    log_processing_step(logger, "Subsetting to trend calculation extent")
    ds_subset = ds_era5.sel(
        longitude=slice(TREND_EXTENT['lon_min'], TREND_EXTENT['lon_max']),
        latitude=slice(TREND_EXTENT['lat_max'], TREND_EXTENT['lat_min'])
    )
    logger.info(f"Subset grid: {len(ds_subset.latitude)} x {len(ds_subset.longitude)}")
    
    # Extract March data
    log_processing_step(logger, "Extracting March sea ice concentration")
    sic_march = ds_subset['siconc'].where(ds_subset.time.dt.month == 3).dropna(dim='time', how='all')
    logger.info(f"March timesteps: {len(sic_march.time)}")
    
    # Calculate spatial trends for Period 1
    log_processing_step(logger, f"Period 1 ({PERIOD_1[0]}-{PERIOD_1[1]}) spatial trends")
    sic_p1 = sic_march.sel(time=slice(f'{PERIOD_1[0]}-01-01', f'{PERIOD_1[1]}-12-31'))
    logger.info(f"Timesteps: {len(sic_p1.time)}")
    slope_p1, pval_p1, r2_p1 = calculate_spatial_trends_vectorized(sic_p1)
    
    # Save processed data
    slope_p1_file = PROCESSED_DIR / 'spatial_trends_period1.nc'
    slope_p1.to_netcdf(slope_p1_file)
    log_output_file(logger, 'processed_data', slope_p1_file)
    
    # Calculate spatial trends for Period 2
    log_processing_step(logger, f"Period 2 ({PERIOD_2[0]}-{PERIOD_2[1]}) spatial trends")
    sic_p2 = sic_march.sel(time=slice(f'{PERIOD_2[0]}-01-01', f'{PERIOD_2[1]}-12-31'))
    logger.info(f"Timesteps: {len(sic_p2.time)}")
    slope_p2, pval_p2, r2_p2 = calculate_spatial_trends_vectorized(sic_p2)
    
    # Save processed data
    slope_p2_file = PROCESSED_DIR / 'spatial_trends_period2.nc'
    slope_p2.to_netcdf(slope_p2_file)
    log_output_file(logger, 'processed_data', slope_p2_file)
    
    # ========================================================================
    # PART 2: LOAD ADDITIONAL DATA FOR PLOTTING
    # ========================================================================
    
    log_processing_step(logger, "PART 2: Loading additional data for plotting")
    
    # Load ETOPO1 bathymetry
    log_data_loading(logger, 'ETOPO1', 'rockhound.fetch_etopo1')
    etopo = rh.fetch_etopo1(version="ice")
    etopo_subset = etopo.sel(longitude=slice(-75, 75), latitude=slice(60, 90))
    
    # Interpolate to higher resolution
    res = 0.1
    new_lons = np.arange(etopo_subset.longitude.values[0],
                        etopo_subset.longitude.values[-1] + res, res)
    new_lats = np.arange(etopo_subset.latitude.values[0],
                        etopo_subset.latitude.values[-1] + res, res)
    etopo_interp = etopo_subset.interp(longitude=new_lons, latitude=new_lats)
    
    # Extract March SIC for the two max extent years
    logger.info(f"Extracting March {MAX_YEAR_P1} and {MAX_YEAR_P2} SIC")
    sic_1986 = ds_era5['siconc'].sel(time=f'{MAX_YEAR_P1}-03-01', method='nearest')
    sic_2023 = ds_era5['siconc'].sel(time=f'{MAX_YEAR_P2}-03-01', method='nearest')
    
    # Mask trends where SIC < 15%
    slope_p1_masked = slope_p1.where(sic_1986 >= SIC_THRESHOLD)
    slope_p2_masked = slope_p2.where(sic_2023 >= SIC_THRESHOLD)
    
    # Load OSISAF data for timeseries
    log_data_loading(logger, 'OSISAF', OSISAF_PATH)
    f_list = os.listdir(OSISAF_PATH)
    
    sie_list = []
    for f in f_list:
        ds = xr.open_dataset(OSISAF_PATH + f)
        ds_monthly = ds.drop_vars('area').resample(time='MS').mean()
        ds_monthly = add_months(ds_monthly)
        
        f_info = f.split('_')
        region = f_info[1]
        title_info = ds.title.split()
        for i, word in enumerate(title_info):
            if word == 'Ice':
                region_name = ' '.join(title_info[1:i-1])
        
        sie = ds_monthly.sie
        sie = sie.rename(region)
        sie.attrs['title'] = ds.title
        sie.attrs['region_name'] = region_name
        sie.attrs['summary'] = ds.summary
        sie_list.append(sie)
    
    ds_nh = fetch_sie('nh', sie_list)
    ds_gs = fetch_sie('fram', sie_list)
    
    # Select season
    if CONFIG['season'] == 'winter':
        season_month = 3
        season_name = 'March'
    else:
        season_month = 9
        season_name = 'September'
    
    logger.info(f"Season: {season_name} (month {season_month})")
    
    # ========================================================================
    # PART 3: CREATE FIGURE
    # ========================================================================
    
    log_processing_step(logger, "PART 3: Creating 4-panel figure")
    
    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(2, 2, 
                          height_ratios=[1.3, 0.7],
                          hspace=0.32, wspace=0.12,  # Reduced from 0.22 to 0.18
                          left=0.08, right=0.92, top=0.96, bottom=0.05)
    
    # Common settings
    bathy_levels = np.arange(-5000, 5200, 200)
    vmax_trend = 25  # % per decade (slope * 10 years * 100 to get % * 1000)
    
    # ====================================================================
    # PANEL A: 1986 (Period 1) SPATIAL MAP
    # ====================================================================
    
    logger.info("Creating Panel (a): 1986 spatial trends")
    ax_a = fig.add_subplot(gs[0, 0], projection=ccrs.NorthPolarStereo(central_longitude=0))
    ax_a.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax_a.coastlines(resolution='50m', linewidth=0.5, color='black', zorder=5)
    
    # Bathymetry
    relief = ax_a.contourf(etopo_interp.longitude, etopo_interp.latitude, etopo_interp.ice,
                           transform=ccrs.PlateCarree(), levels=bathy_levels,
                           cmap=cmocean.cm.topo, alpha=RELIEF_ALPHA, extend='both', zorder=1)
    
    # Sea ice edge
    sic_1986_masked = sic_1986.where(sic_1986 > 0)
    ax_a.contour(ds_era5.longitude, ds_era5.latitude, sic_1986_masked.values,
                transform=ccrs.PlateCarree(), levels=[SIC_THRESHOLD],
                colors=COLOR_ICE_EDGE, linewidths=4, zorder=4)
    
    # Spatial trends - MODIFIED: convert to % per decade
    trend_plot = ax_a.pcolormesh(
        ds_subset.longitude, ds_subset.latitude,
        slope_p1_masked.values * 1000,  # yr⁻¹ * 10 yr * 100 for %
        transform=ccrs.PlateCarree(), cmap=cmap_trend,
        vmin=-vmax_trend, vmax=vmax_trend, alpha=TREND_ALPHA,
        shading='auto', zorder=3
    )
    
    # Greenland Sea boundary
    def plot_segment(ax, v1, v2):
        lons = np.linspace(VERTICES[v1][0], VERTICES[v2][0], 100)
        lats = np.linspace(VERTICES[v1][1], VERTICES[v2][1], 100)
        ax.plot(lons, lats, color=COLOR_BOUNDARY, linestyle='--',
               linewidth=1.5, transform=ccrs.PlateCarree(), zorder=10)
    
    for v1, v2 in [('NW','NE'), ('SW','SE'), ('SE','NE'), ('SW','W_Mid'), ('W_Mid','NW')]:
        plot_segment(ax_a, v1, v2)
    
    # # Fram Strait moorings
    # mooring_lons = list(MOORINGS.values())
    # mooring_lats = [MOORING_LAT] * len(MOORINGS)
    # ax_a.plot(mooring_lons, mooring_lats, marker='o', markersize=4,
    #          color=COLOR_MOORING, linestyle='none',
    #          markeredgecolor='darkviolet', markeredgewidth=0.6,
    #          transform=ccrs.PlateCarree(), zorder=11)
    
    # MODIFIED: Gridlines with zorder=20
    gl_a = ax_a.gridlines(draw_labels=True, x_inline=False, y_inline=False,
                         linewidth=0.5, color='gray', alpha=0.85, linestyle=':',
                         zorder=20)
    gl_a.right_labels = False
    gl_a.top_labels = True
    gl_a.xlabel_style = {'size': 10}
    gl_a.ylabel_style = {'size': 10}
    gl_a.xlocator = mticker.FixedLocator(np.arange(-90, 91, 15))
    gl_a.ylocator = mticker.FixedLocator(np.arange(60, 91, 4))
    gl_a.xformatter = LongitudeFormatter()
    gl_a.yformatter = LatitudeFormatter()
    
    ax_a.set_title(f'a) March {MAX_YEAR_P1} ice edge, {PERIOD_1[0]}-{PERIOD_1[1]} trend',
                  loc='left', fontsize=11)
    
    # ====================================================================
    # PANEL B: 2023 (Period 2) SPATIAL MAP
    # ====================================================================
    
    logger.info("Creating Panel (b): 2023 spatial trends")
    ax_b = fig.add_subplot(gs[0, 1], projection=ccrs.NorthPolarStereo(central_longitude=0))
    ax_b.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax_b.coastlines(resolution='50m', linewidth=0.5, color='black', zorder=5)
    
    # Bathymetry
    ax_b.contourf(etopo_interp.longitude, etopo_interp.latitude, etopo_interp.ice,
                 transform=ccrs.PlateCarree(), levels=bathy_levels,
                 cmap=cmocean.cm.topo, alpha=RELIEF_ALPHA, extend='both', zorder=1)
    
    # Sea ice edge
    sic_2023_masked = sic_2023.where(sic_2023 > 0)
    ax_b.contour(ds_era5.longitude, ds_era5.latitude, sic_2023_masked.values,
                transform=ccrs.PlateCarree(), levels=[SIC_THRESHOLD],
                colors=COLOR_ICE_EDGE, linewidths=4, zorder=4)
    
    # Spatial trends - MODIFIED: convert to % per decade
    ax_b.pcolormesh(
        ds_subset.longitude, ds_subset.latitude,
        slope_p2_masked.values * 1000,  # yr⁻¹ * 10 yr * 100 for %
        transform=ccrs.PlateCarree(), cmap=cmap_trend,
        vmin=-vmax_trend, vmax=vmax_trend, alpha=TREND_ALPHA,
        shading='auto', zorder=3
    )
    
    # Greenland Sea boundary and moorings
    for v1, v2 in [('NW','NE'), ('SW','SE'), ('SE','NE'), ('SW','W_Mid'), ('W_Mid','NW')]:
        plot_segment(ax_b, v1, v2)
    
    # ax_b.plot(mooring_lons, mooring_lats, marker='o', markersize=4,
    #          color=COLOR_MOORING, linestyle='none',
    #          markeredgecolor='darkviolet', markeredgewidth=0.6,
    #          transform=ccrs.PlateCarree(), zorder=11)
    
    # MODIFIED: Gridlines with zorder=20
    gl_b = ax_b.gridlines(draw_labels=True, x_inline=False, y_inline=False,
                         linewidth=0.5, color='gray', alpha=0.85, linestyle=':',
                         zorder=20)
    gl_b.right_labels = False
    gl_b.top_labels = True
    gl_b.xlabel_style = {'size': 10}
    gl_b.ylabel_style = {'size': 10}
    gl_b.xlocator = mticker.FixedLocator(np.arange(-90, 91, 15))
    gl_b.ylocator = mticker.FixedLocator(np.arange(60, 91, 4))
    gl_b.xformatter = LongitudeFormatter()
    gl_b.yformatter = LatitudeFormatter()
    
    ax_b.set_title(f'b) March {MAX_YEAR_P2} ice edge, {PERIOD_2[0]}-{PERIOD_2[1]} trend',
                  loc='left', fontsize=11)
    
    # ====================================================================
    # ADD COLORBARS
    # ====================================================================
    
    # Bathymetry colorbar under panel a (left)
    cbar_ax1 = fig.add_axes([0.13, 0.4, 0.3, 0.014])  # Adjusted y position
    cbar1 = plt.colorbar(relief, cax=cbar_ax1, orientation='horizontal')
    cbar1.set_label('Elevation (m)', fontsize=9)
    cbar1.ax.tick_params(labelsize=8)
    
    # SIC trend colorbar under panel b (right)
    cbar_ax2 = fig.add_axes([0.57, 0.4, 0.3, 0.014])  # Adjusted y position
    cbar2 = plt.colorbar(trend_plot, cax=cbar_ax2, orientation='horizontal', extend='both')
    cbar2.set_label('SIC trend (% decade⁻¹)', fontsize=9)
    cbar2.ax.tick_params(labelsize=8)
    
    # ====================================================================
    # PANEL C: PAN-ARCTIC TIMESERIES
    # ====================================================================
    
    logger.info("Creating Panel (c): Pan-Arctic timeseries")
    ax_c = fig.add_subplot(gs[1, 0])
    
    # Breakpoint for pan-Arctic
    if CONFIG['season'] == 'winter':
        break_year = '2017'
    else:
        break_year = '2012'
    
    split_1 = slice('1979', break_year)
    split_2 = slice(break_year, '2025')
    
    x_1 = ds_nh.time.sel(time=split_1).where(ds_nh.month == season_month, drop=True)
    x_num_1 = dates.date2num(x_1)
    x_2 = ds_nh.time.sel(time=split_2).where(ds_nh.month == season_month, drop=True)
    x_num_2 = dates.date2num(x_2)
    
    y_1 = ds_nh.sel(time=split_1).where(ds_nh.month == season_month, drop=True)
    p_1, trend_1, stats_1 = get_trend(x_num_1, y_1)
    y_2 = ds_nh.sel(time=split_2).where(ds_nh.month == season_month, drop=True)
    p_2, trend_2, stats_2 = get_trend(x_num_2, y_2)
    
    # Mann-Kendall test
    mk_1 = mann_kendall_test(x_num_1, y_1)
    mk_2 = mann_kendall_test(x_num_2, y_2)
    
    logger.info(f"Period 1 (1979-{break_year}): Trend={trend_1*10:.3f}, MK τ={mk_1['tau']:.3f}")
    logger.info(f"Period 2 ({break_year}-2024): Trend={trend_2*10:.3f}, MK τ={mk_2['tau']:.3f}")
    
    # Uncertainty bands
    if CONFIG['show_uncertainty_bands']:
        if CONFIG['uncertainty_method'] == 'bootstrap':
            ci_1 = bootstrap_trend_ci(x_num_1, y_1, n_bootstrap=CONFIG['n_bootstrap'],
                                     confidence=CONFIG['confidence_level'])
            ci_2 = bootstrap_trend_ci(x_num_2, y_2, n_bootstrap=CONFIG['n_bootstrap'],
                                     confidence=CONFIG['confidence_level'])
        
        ax_c.fill_between(x_1, ci_1['lower_bound'], ci_1['upper_bound'],
                         color=COLOR_ARCTIC, alpha=CI_ALPHA, zorder=1)
        ax_c.fill_between(x_2, ci_2['lower_bound'], ci_2['upper_bound'],
                         color=COLOR_ARCTIC, alpha=CI_ALPHA, zorder=1)
    
    # Plot data - GRAY
    x_all = ds_nh.time.where(ds_nh.month == season_month, drop=True)
    y_all = ds_nh.where(ds_nh.month == season_month, drop=True)
    
    ax_c.plot(x_all, y_all, color=COLOR_DATA, linewidth=1, alpha=0.5, zorder=2)
    ax_c.scatter(x_all, y_all, s=20, color=COLOR_DATA, alpha=0.6, zorder=3, edgecolors='none')
    
    # Plot trends - PURPLE
    ax_c.plot(x_1, p_1(x_num_1), color=COLOR_ARCTIC, linewidth=2.5, 
             linestyle='--', zorder=4)
    ax_c.plot(x_2, p_2(x_num_2), color=COLOR_ARCTIC, linewidth=2.5,
             linestyle='--', zorder=4)
    
    # Breakpoint line
    ax_c.axvline(x=pd.Timestamp(f'{break_year}-01-01'), color=COLOR_BREAKPOINT,
                linestyle=':', linewidth=1.5, alpha=0.7, zorder=1)
    
    # Add stats box
    stats_text = (
        f"1979-{break_year}  ***\n"
        f"  {trend_1*10:.2f} ×10⁶ km² decade⁻¹\n"
        f"  R² = {stats_1['r_squared']:.3f}\n\n"
        f"{break_year}-2025  n.s.\n"
        f"  {trend_2*10:+.2f} ×10⁶ km² decade⁻¹\n"
        f"  R² = {stats_2['r_squared']:.3f}"
    )
    
    ax_c.text(0.98, 0.98, stats_text, transform=ax_c.transAxes,
             fontsize=8, verticalalignment='top', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray', linewidth=0.5),
             family='monospace')
    
    ax_c.set_xlabel('Year', fontsize=10)
    ax_c.set_ylabel(r'Sea ice extent ($\times10^{6}$ km$^{2}$)', fontsize=10)
    ax_c.set_title(f'c) Pan-Arctic {season_name} SIE', loc='left', fontsize=11)
    ax_c.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    
    # ====================================================================
    # PANEL D: GREENLAND SEA TIMESERIES
    # ====================================================================
    
    logger.info("Creating Panel (d): Greenland Sea timeseries")
    ax_d = fig.add_subplot(gs[1, 1])
    
    # Use 2015 breakpoint for Greenland Sea
    break_year_gs = str(BREAKPOINT_YEAR)
    
    split_1 = slice('1979', break_year_gs)
    split_2 = slice(break_year_gs, '2025')
    
    x_1 = ds_gs.time.sel(time=split_1).where(ds_gs.month == season_month, drop=True)
    x_num_1 = dates.date2num(x_1)
    x_2 = ds_gs.time.sel(time=split_2).where(ds_gs.month == season_month, drop=True)
    x_num_2 = dates.date2num(x_2)
    
    y_1 = ds_gs.sel(time=split_1).where(ds_gs.month == season_month, drop=True)
    p_1, trend_1, stats_1 = get_trend(x_num_1, y_1)
    y_2 = ds_gs.sel(time=split_2).where(ds_gs.month == season_month, drop=True)
    p_2, trend_2, stats_2 = get_trend(x_num_2, y_2)
    
    # Mann-Kendall test
    mk_1 = mann_kendall_test(x_num_1, y_1)
    mk_2 = mann_kendall_test(x_num_2, y_2)
    trend_comp = compare_trends(x_num_1, y_1, x_num_2, y_2)
    
    logger.info(f"GS Period 1: Trend={trend_1*10:.3f}, MK τ={mk_1['tau']:.3f}")
    logger.info(f"GS Period 2: Trend={trend_2*10:.3f}, MK τ={mk_2['tau']:.3f}")
    logger.info(f"Trends significantly different: {trend_comp['significant']}")
    
    # Uncertainty bands
    if CONFIG['show_uncertainty_bands']:
        if CONFIG['uncertainty_method'] == 'bootstrap':
            ci_1 = bootstrap_trend_ci(x_num_1, y_1, n_bootstrap=CONFIG['n_bootstrap'],
                                     confidence=CONFIG['confidence_level'])
            ci_2 = bootstrap_trend_ci(x_num_2, y_2, n_bootstrap=CONFIG['n_bootstrap'],
                                     confidence=CONFIG['confidence_level'])
        
        ax_d.fill_between(x_1, ci_1['lower_bound'], ci_1['upper_bound'],
                         color=COLOR_GS, alpha=CI_ALPHA, zorder=1)
        ax_d.fill_between(x_2, ci_2['lower_bound'], ci_2['upper_bound'],
                         color=COLOR_GS, alpha=CI_ALPHA, zorder=1)
    
    # Plot data - GRAY
    x_all = ds_gs.time.where(ds_gs.month == season_month, drop=True)
    y_all = ds_gs.where(ds_gs.month == season_month, drop=True)
    
    ax_d.plot(x_all, y_all, color=COLOR_DATA, linewidth=1, alpha=0.5, zorder=2)
    ax_d.scatter(x_all, y_all, s=20, color=COLOR_DATA, alpha=0.6, zorder=3, edgecolors='none')
    
    # Plot trends - RED
    ax_d.plot(x_1, p_1(x_num_1), color=COLOR_GS, linewidth=2.5,
             linestyle='--', zorder=4)
    ax_d.plot(x_2, p_2(x_num_2), color=COLOR_GS, linewidth=2.5,
             linestyle='--', zorder=4)
    
    # Red squares for max extent years
    for year in [MAX_YEAR_P1, MAX_YEAR_P2]:
        year_data = y_all.where(x_all.dt.year == year, drop=True)
        if len(year_data) > 0:
            year_time = x_all.where(x_all.dt.year == year, drop=True).values[0]
            ax_d.scatter(year_time, year_data.values[0], s=120, marker='s',
                        color='#C62828', edgecolors='none',  # Match trend color
                        zorder=5, alpha=0.9)
    
    # Breakpoint line
    ax_d.axvline(x=pd.Timestamp(f'{break_year_gs}-01-01'), color=COLOR_BREAKPOINT,
                linestyle=':', linewidth=1.5, alpha=0.7, zorder=1)
    
    # Add stats box
    stats_text = (
        f"1979-{PERIOD_1[1]}  ***\n"
        f"  {trend_1*10:.2f} ×10⁶ km² decade⁻¹\n"
        f"  R² = {stats_1['r_squared']:.3f}\n\n"
        f"{PERIOD_2[0]}-2025  n.s.\n"
        f"  {trend_2*10:+.2f} ×10⁶ km² decade⁻¹\n"
        f"  R² = {stats_2['r_squared']:.3f}\n\n"
        f"Trends differ: p={trend_comp['p_value']:.3f}"
        )
        
    ax_d.text(0.98, 0.98, stats_text, transform=ax_d.transAxes,
             fontsize=8, verticalalignment='top', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray', linewidth=0.5),
             family='monospace')
    
    ax_d.set_xlabel('Year', fontsize=10)
    ax_d.set_title(f'd) Greenland Sea {season_name} SIE', loc='left', fontsize=11)
    ax_d.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    
    # ========================================================================
    # SAVE FIGURE
    # ========================================================================
    
    plt.tight_layout()
    
    output_file = OUTPUT_DIR / 'figure_01.png'
    fig.savefig(output_file, dpi=DPI, bbox_inches='tight', facecolor='white')
    logger.info(f"Figure saved: {output_file}")
    log_output_file(logger, 'figure', output_file)
    
    plt.show()
    
    # ========================================================================
    # GENERATE METHODS DOCUMENTATION
    # ========================================================================
    
    methods_file = METHODS_DIR / 'figure_01_methods.md'
    with open(methods_file, 'w') as f:
        f.write("""# Figure 1: Methodology

## Spatial Sea Ice Concentration Trends

### Data Sources
- **ERA5 Reanalysis**: Monthly sea ice concentration from ECMWF ERA5 atmospheric reanalysis (Hersbach et al., 2020)
- **Temporal Coverage**: 1979-2025
- **Spatial Resolution**: 0.25° × 0.25°
- **Domain**: Arctic region (60°N-90°N, 90°W-90°E)

### Processing Methods
Sea ice concentration (SIC) data were extracted for March (month of maximum extent) and subset to the Arctic domain. Linear trends were calculated at each grid cell using ordinary least squares regression:

SIC_trend = β₀ + β₁ × year

where β₁ represents the trend in yr⁻¹. Trends were converted to % decade⁻¹ by multiplying by 1000 (10 years × 100 for percentage).

Only grid cells with at least 5 valid observations were included in the trend analysis. Statistical significance was assessed using a two-tailed t-test (α = 0.05).

**Two temporal periods were analyzed:**
- Period 1: 1979-2014 (36 years)
- Period 2: 2015-2025 (11 years)

Spatial trends were masked where March SIC < 15% (ice edge threshold) to focus on ice-covered regions.

### Bathymetry
ETOPO1 ice surface elevation data (Amante & Eakins, 2009) were interpolated to 0.1° resolution for background context.

## Sea Ice Extent Timeseries

### Data Sources
- **OSI SAF Sea Ice Index**: Regional sea ice extent from EUMETSAT Ocean and Sea Ice Satellite Application Facility (OSI SAF, 2017)
- **Regions**: Northern Hemisphere (pan-Arctic) and Fram Strait/Greenland Sea
- **Temporal Coverage**: 1979-2025
- **Temporal Resolution**: Daily, aggregated to monthly means

### Statistical Methods

#### Split Linear Trends
March sea ice extent timeseries were analyzed using split linear trends with predetermined breakpoints:
- **Pan-Arctic**: 2017 (following method of Serreze & Meier, 2019)
- **Greenland Sea**: 2015 (identified from piecewise regression analysis, see Figure S1)

For each period, linear trends were calculated using ordinary least squares regression. Trend significance was assessed using:

1. **Parametric test**: Two-tailed t-test on regression slope (α = 0.05)
2. **Non-parametric test**: Mann-Kendall trend test (Mann, 1945; Kendall, 1975)

The Mann-Kendall test evaluates monotonic trends without assuming linearity or normality. The test statistic S is:

S = Σᵢ₌₁ⁿ⁻¹ Σⱼ₌ᵢ₊₁ⁿ sign(xⱼ - xᵢ)

Kendall's tau (τ) measures correlation:

τ = S / [n(n-1)/2]

#### Uncertainty Quantification
Bootstrap confidence intervals (95%) were calculated for trend estimates using 1000 resamples with replacement (Efron & Tibshirani, 1993). For each bootstrap iteration:
1. Resample time-extent pairs with replacement
2. Fit linear regression
3. Store slope and prediction

Confidence intervals were derived from the 2.5th and 97.5th percentiles of the bootstrap distribution.

#### Trend Comparison
Differences between Period 1 and Period 2 trends were tested using the Chow test (Chow, 1960), which compares:
- H₀: Single pooled trend fits both periods
- H₁: Separate trends are significantly different

The F-statistic tests whether the reduction in residual sum of squares justifies the additional parameters.

## References

Amante, C., & Eakins, B. W. (2009). ETOPO1 1 Arc-Minute Global Relief Model: Procedures, Data Sources and Analysis. NOAA Technical Memorandum NESDIS NGDC-24.

Chow, G. C. (1960). Tests of equality between sets of coefficients in two linear regressions. *Econometrica*, 28(3), 591-605.

Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall/CRC.

Hersbach, H., et al. (2020). The ERA5 global reanalysis. *Quarterly Journal of the Royal Meteorological Society*, 146(730), 1999-2049. https://doi.org/10.1002/qj.3803

Kendall, M. G. (1975). *Rank Correlation Methods* (4th ed.). Charles Griffin.

Mann, H. B. (1945). Nonparametric tests against trend. *Econometrica*, 13(3), 245-259.

OSI SAF (2017). Global Sea Ice Concentration Climate Data Record v2.0 - Multimission. EUMETSAT SAF on Ocean and Sea Ice. https://osi-saf.eumetsat.int/

Serreze, M. C., & Meier, W. N. (2019). The Arctic's sea ice cover: trends, variability, predictability, and comparisons to the Antarctic. *Annals of the New York Academy of Sciences*, 1436(1), 36-53.

## Software

Analysis performed using:
- Python 3.10.12
- xarray 2024.x for NetCDF data handling
- SciPy for statistical tests
- Matplotlib and Cartopy for visualization
- Rockhound for bathymetry data access
""")
    
    logger.info(f"Methods documentation saved: {methods_file}")
    log_output_file(logger, 'methods', methods_file)
    
    # Log completion
    log_completion(logger, start_time)
    
    logger.info("="*70)
    logger.info("FIGURE 1 COMPLETE")
    logger.info("="*70)

except Exception as e:
    log_error(logger, e, context="During Figure 1 generation")
    raise

INFO     | ======================================================================
INFO     | Starting analysis: figure_01
INFO     | Log file: outputs/logs/figure_01_20260227_231237.log
INFO     | Timestamp: 2026-02-27 23:12:37
INFO     | ======================================================================
INFO     | Loading configuration...
INFO     | Periods: (1979, 2016) and (2016, 2025)
INFO     | Breakpoint year: 2016
INFO     | Functions defined
INFO     | Processing: PART 1: Calculating Spatial Trends
INFO     | Loading ERA5 data from: ../../era5/era5_*_Arctic.nc
INFO     | Time range: 1979 to 2025
INFO     | Grid: 361 x 1440
INFO     | Processing: Subsetting to trend calculation extent
INFO     | Subset grid: 121 x 721
INFO     | Processing: Extracting March sea ice concentration
INFO     | March timesteps: 47
INFO     | Processing: Period 1 (1979-2016) spatial trends
INFO     | Timesteps: 38
INFO     | Created processed_data: spatial_trends_period1.nc
INFO     | Processing: 